# Simple Relative Strength Index Recreation


#### Objectives:
Recreate the relative strength index and apply Wilder's smoothing on one year of forex data.

In [ ]:
#!pip install yfinance
#!pip install mplfinance
import pandas as pd
import numpy as np
import yfinance as yf
import mplfinance as mpf

## Pulling Data from yfinance & Data Cleaning

* 258 days of price close data from 1st July 2025 to 29th June 2026 (1 year) of EURUSD

In [ ]:
fx_ticker = "EURUSD=X"
eur_usd_data = yf.download(fx_ticker, start="2025-07-01", end="2026-06-30")

print("Duplicate columns?", eur_usd_data.columns.duplicated().any())
print("Column dtypes:\n", eur_usd_data[['Open','High','Low','Close']].dtypes)
print("Sample Open values:\n", eur_usd_data['Open'].head())

[*********************100%***********************]  1 of 1 completed

Duplicate columns? False
Column dtypes:
 Price  Ticker  
Open   EURUSD=X    float64
High   EURUSD=X    float64
Low    EURUSD=X    float64
Close  EURUSD=X    float64
dtype: object
Sample Open values:
 Ticker      EURUSD=X
Date                
2025-07-01  1.178717
2025-07-02  1.180554
2025-07-03  1.180025
2025-07-04  1.177149
2025-07-07  1.178078


In [ ]:
eur_usd_close = eur_usd_data['Close']
eur_usd_close

Ticker,EURUSD=X
Date,
2025-07-01,1.178717
2025-07-02,1.180554
2025-07-03,1.180025
2025-07-04,1.177149
2025-07-07,1.178078
...,...
2026-06-23,1.142714
2026-06-24,1.137954
2026-06-25,1.135422


## Calculating Relative Strength Index (RSI)

### What is the Relative Strength Index? 

The RSI is a momentum indicator which looks at how much recent price movement has been in one direction vs the other. 

Simply put: 
* If gains dominates = momentum is bullish
* If losses dominates = momentum is bearish

**Relative Strength Formula (RS)** = Average Gain / Average Loss 
* RS a ratio which shows how much more gains than losses over a period

**Relative Strength Index Formula (RSI)** = 100 - (100 - (1 + RS))
* Scales RS to 0 to 100 values
* Simply shows what % of price movements was upwards


To increase stability and prevent sharp drops in old data in our indicator we apply Wilder's Smoothing. 

We first obtain the seed for the first value using the first 15 days of price changes (Day 1 has no gain or loss)

In [ ]:
def change(price_Ytd, price_Tdy):
    return price_Tdy - price_Ytd

eur_usd = eur_usd_close.copy()
eur_usd['Gain'] = 0.0
eur_usd['Loss'] = 0.0

# Calculate price change for entire dataset
for i in range(1, len(eur_usd)):
    change_in_price = change(eur_usd['EURUSD=X'].iloc[i - 1], eur_usd['EURUSD=X'].iloc[i])
    current_date = eur_usd.index[i]
    if change_in_price > 0:
        eur_usd.loc[current_date, 'Gain'] = change_in_price
    elif change_in_price < 0:
        eur_usd.loc[current_date, 'Loss'] = -1 * change_in_price

print(eur_usd.head(15))

# first 15 days (day 0 has no gain or loss)
seed_average_gain = eur_usd['Gain'].iloc[1:15].sum() / 14
seed_average_loss = eur_usd['Loss'].iloc[1:15].sum() / 14

# Calculate Relative Strength index
def calculate_RSI(avg_gain, avg_loss):
    return 100 - (100 / (1 + avg_gain / avg_loss))

# Seed the first average RS values
eur_usd['RSI'] = np.nan
eur_usd.iloc[14, eur_usd.columns.get_loc('RSI')] = calculate_RSI(seed_average_gain, seed_average_loss)
eur_usd.head(15)

Ticker      EURUSD=X      Gain      Loss
Date                                    
2025-07-01  1.178717  0.000000  0.000000
2025-07-02  1.180554  0.001837  0.000000
2025-07-03  1.180025  0.000000  0.000529
2025-07-04  1.177149  0.000000  0.002875
2025-07-07  1.178078  0.000929  0.000000
2025-07-08  1.173654  0.000000  0.004424
2025-07-09  1.172457  0.000000  0.001197
2025-07-10  1.173117  0.000660  0.000000
2025-07-11  1.170275  0.000000  0.002842
2025-07-14  1.168211  0.000000  0.002064
2025-07-15  1.166630  0.000000  0.001581
2025-07-16  1.160739  0.000000  0.005891
2025-07-17  1.163575  0.002836  0.000000
2025-07-18  1.161548  0.000000  0.002027
2025-07-21  1.163075  0.001526  0.000000


Ticker,EURUSD=X,Gain,Loss,RSI
Date,,,,
2025-07-01,1.178717,0.000000,0.000000,NaN
2025-07-02,1.180554,0.001837,0.000000,NaN
2025-07-03,1.180025,0.000000,0.000529,NaN
2025-07-04,1.177149,0.000000,0.002875,NaN
2025-07-07,1.178078,0.000929,0.000000,NaN
2025-07-08,1.173654,0.000000,0.004424,NaN
2025-07-09,1.172457,0.000000,0.001197,NaN
2025-07-10,1.173117,0.000660,0.000000,NaN
2025-07-11,1.170275,0.000000,0.002842,NaN


Wilder's Smoothing is a technique which reduces short term market noise by allowing the data to react slower to price changes via a recursive formula.

Avg Gain = (Gain (Yesterday) * (n - 1) + Gain (Today)) / 14

In [ ]:
# Apply Wilder's Smoothing
def calculate_Average_Price(prev_price, current_price):
    return ((prev_price * 13) + current_price) / 14

# Day 16
rolling_gain = calculate_Average_Price(seed_average_gain, eur_usd['Gain'].iloc[15])
rolling_loss = calculate_Average_Price(seed_average_loss, eur_usd['Loss'].iloc[15])
eur_usd.iloc[15, eur_usd.columns.get_loc('RSI')] = calculate_RSI(rolling_gain, rolling_loss)

# Calculate the RSI for all values (Day 17 onwards)
for index in range(16, len(eur_usd)):
    rolling_gain = calculate_Average_Price(rolling_gain, eur_usd['Gain'].iloc[index])
    rolling_loss = calculate_Average_Price(rolling_loss, eur_usd['Loss'].iloc[index])
    eur_usd.iloc[index, eur_usd.columns.get_loc('RSI')] = calculate_RSI(rolling_gain, rolling_loss)
    
eur_usd.tail(5)

Ticker,EURUSD=X,Gain,Loss,RSI
Date,,,,
2026-06-23,1.142714,0.000000,0.003563,29.576397
2026-06-24,1.137954,0.000000,0.004759,26.172772
2026-06-25,1.135422,0.000000,0.002532,24.553580
2026-06-26,1.136170,0.000748,0.000000,26.009889
2026-06-29,1.138563,0.002393,0.000000,30.623154


**Graphical Representation**

In [ ]:
ohlc = eur_usd_data.copy()

# Flatten the MultiIndex columns (Price, Ticker)
if isinstance(ohlc.columns, pd.MultiIndex):
    ohlc.columns = ohlc.columns.get_level_values(0)

#Create 30 / 70 lines in the chart
overbought = pd.Series(70, index = ohlc.index)
oversold = pd.Series(30, index = ohlc.index)

#Attach RSI values
ohlc['RSI'] = eur_usd['RSI']

#RSI Panel
rsi_panel = mpf.make_addplot(
    ohlc['RSI'],
    panel=1,
    color='purple',
    ylabel='RSI',
    ylim=(0, 100)
)

ob_line = mpf.make_plot(overbought, panel=1, color='red', linestyle='--', width =0.8)
ob_line = mpf.make_plot(overbought, panel=1, color='green', linestyle='--', width =0.8)

#Historical Price Plot
mpf.plot(
    ohlc,
    type='candle',
    style='yahoo',
    addplot=[rsi_panel, ob_line, os_line],
    panel_ratios=(3, 1),
    volume=True,
    title='EUR/USD with RSI',
    figsize=(14,8))

AttributeError: module 'mplfinance' has no attribute 'make_plot'